# RadixSort

**Radix** = baza de numerație. Ideea: sortăm numerele **cifră cu cifră**, cu un CountSort **stabil** pe fiecare cifră, începând de la cifra **cea mai puțin semnificativă** (unități, apoi zeci, sute, ...).

Este exact „sortarea după mai multe criterii” din `countsort.ipynb`: cifra unităților e criteriul cel mai puțin important, iar prima cifră e cel mai important.

Notații: n = numărul de elemente, b = baza, d = numărul de cifre ale maximului în baza b, adică ⌈log_b(max + 1)⌉.

Proprietăți:
- d treceri, fiecare un CountSort cu k = b ⇒ **O(d · (n + b))**
- memorie suplimentară **O(n + b)**
- **stabil**
- o bază mai mare înseamnă mai puține treceri, dar un vector de frecvențe mai mare

## 1. Cifra de pe poziția `exp`

În baza 10: cifra unităților este `(x // 1) % 10`, a zecilor `(x // 10) % 10`, a sutelor `(x // 100) % 10` etc.

In [ ]:
x = 802
for exp in [1, 10, 100, 1000]:
    print(f"exp = {exp:>4}: cifra = {(x // exp) % 10}")

## 2. RadixSort în baza 10

In [ ]:
# countsort stabil dupa cifra (x // exp) % base
def countsort_digit(a, exp, base=10):
    f = [0] * base
    for x in a:
        f[(x // exp) % base] += 1
    for i in range(1, base):
        f[i] += f[i - 1]
    out = [0] * len(a)
    for x in reversed(a):              # de la final => stabil
        d = (x // exp) % base
        out[f[d] - 1] = x
        f[d] -= 1
    return out


# doar pentru numere naturale
def radixsort(a, base=10, verbose=False):
    if not a:
        return []
    m = max(a)
    exp = 1
    while m // exp > 0:
        a = countsort_digit(a, exp, base)
        if verbose:
            print(f"[exp = {exp:>3}]  {a}")
        exp *= base
    return a


v = [170, 45, 75, 90, 802, 24, 2, 66]
print(f"inițial:      {v}")
radixsort(v, verbose=True)

După trecerea `exp = 10`, numerele sunt sortate după **ultimele 2 cifre**. După `exp = 100`, sunt sortate după ultimele 3 cifre, adică complet.

**Invariant**: după trecerea i, vectorul e sortat după ultimele i cifre. De ce? La trecerea i sortăm după cifra i. Dacă două numere au aceeași cifră i, CountSort-ul **stabil** le lasă în ordinea de dinainte, adică ordonate după ultimele i − 1 cifre.

## 3. De ce trebuie CountSort stabil?

Dacă pe fiecare cifră folosim un CountSort **nestabil** (parcurgere de la început), fiecare trecere strică ordinea lăsată de trecerile anterioare:

In [ ]:
def countsort_digit_unstable(a, exp, base=10):
    f = [0] * base
    for x in a:
        f[(x // exp) % base] += 1
    for i in range(1, base):
        f[i] += f[i - 1]
    out = [0] * len(a)
    for x in a:                        # de la inceput => nu e stabil
        d = (x // exp) % base
        out[f[d] - 1] = x
        f[d] -= 1
    return out


a = [170, 45, 75, 90, 802, 24, 2, 66]
exp = 1
while max(a) // exp > 0:
    a = countsort_digit_unstable(a, exp)
    print(f"[exp = {exp:>3}]  {a}")
    exp *= 10
print("rezultat:", a, "<- greșit!")

## 4. Alegerea bazei

Avem d = ⌈log_b(max + 1)⌉ treceri, fiecare O(n + b).

| baza b | treceri pentru max < 2³² | mărimea vectorului de frecvențe |
|---|---|---|
| 2 | 32 | 2 |
| 10 | 10 | 10 |
| 2⁸ = 256 | 4 | 256 |
| 2¹⁶ = 65536 | 2 | 65536 |

Dacă baza este o putere a lui 2, cifra se obține cu operații pe biți: în baza 256, cifra i este `(x >> (8 * i)) & 255`. Așa se face RadixSort în practică (vezi `radixsort.cpp`).

Pentru întregi pe 32 de biți cu b = 256 avem d = 4 constant, deci complexitatea este **O(n)**.

In [ ]:
# baza 256, cifrele le scoatem cu operatii pe biti (doar numere naturale)
def radixsort_256(a):
    if not a:
        return []
    m = max(a)
    shift = 0
    while m >> shift > 0:
        f = [0] * 256
        for x in a:
            f[(x >> shift) & 255] += 1
        for i in range(1, 256):
            f[i] += f[i - 1]
        out = [0] * len(a)
        for x in reversed(a):
            d = (x >> shift) & 255
            out[f[d] - 1] = x
            f[d] -= 1
        a = out
        shift += 8
    return a


# exemplul la care countsort avea nevoie de un vector de frecvente de 400 de milioane
print(radixsort_256([400000000, 5, 8, 123456789, 70000]))

## 5. Numere negative

Varianta simplă: translatăm cu minimul (ca la CountSort), sortăm, apoi translatăm înapoi.

In [ ]:
def radixsort_any(a):
    if not a:
        return []
    lo = min(a)
    return [x + lo for x in radixsort([x - lo for x in a])]


print(radixsort_any([3, -20, 7, 0, -5, 300, 1]))

## 6. RadixSort vs. sortările prin comparații

- Limita Ω(n log n) din C6 este pentru sortările **prin comparații**. RadixSort nu compară elemente, deci limita nu i se aplică.
- O(d · (n + b)) este mai bun decât O(n log n) când d e mic: numere de lungime fixă (întregi pe 32/64 de biți, CNP-uri, coduri poștale, șiruri de lungime fixă).
- Dezavantaje: merge doar pe chei care se pot descompune în „cifre”, folosește memorie O(n + b) (nu e in-place), iar pentru chei foarte lungi d poate fi mare.

Timpii măsurați în Python nu sunt relevanți aici, pentru că `sorted` e scris în C. Comparația cu `std::sort` este în `radixsort.cpp`.

## Testare

In [ ]:
import random

for _ in range(500):
    arr = [random.randint(0, 10**6) for _ in range(random.randint(0, 50))]
    assert radixsort(arr) == sorted(arr)
    assert radixsort(arr, base=2) == sorted(arr)
    assert radixsort(arr, base=1000) == sorted(arr)
    assert radixsort_256(arr) == sorted(arr)

    neg = [random.randint(-1000, 1000) for _ in range(random.randint(0, 50))]
    assert radixsort_any(neg) == sorted(neg)
print("toate testele au trecut")